In [ ]:
RESULTS_CSV    = 'results.csv'
SCORECARD_XLSX = 'scorecard.xlsx'
INDEX_HTML     = 'index.html'

# default summary by score window 
SUMMARY_BANDS = [
    ( 3.5, "{entity} responded to most requests quickly and in full."),
    ( 2,   "{entity} was broadly cooperative on public records requests."),
    (-0.5, "{entity} shows mixed compliance with the state's right-to-know law."),
    (-2.5, "{entity} is slow to respond and frequently rejects records routinely considered public."),
    (-99,  "{entity} is largely unresponsive to public records requests filed under the state's right-to-know law."),
]

def default_summary(entity: str, score):
    if score is None: return f'{entity} has no graded responses yet.'
    for threshold, template in SUMMARY_BANDS:
        if score >= threshold:
            return template.format(entity=entity)
    return SUMMARY_BANDS[-1][1].format(entity=entity)


DISTRICT_DISPLAY_NAME = {
    'SAU67':                                                 'Bow School District/SAU67',
    'SAU6, serving the communities of Claremont and Unity':  'Claremont School District/SAU6',
    'Derry Cooperative School District':                     'Derry Cooperative School District/SAU10',
    'SAU20':                                                 'Gorham School District/SAU20',
    'SAU70':                                                 'Interstate School District/SAU70',
    'Manchester School District':                            'Manchester School District/SAU37',
    'Monadnock Regional School District':                    'Monadnock Regional School District/SAU93',
    'Nashua School District':                                'Nashua School District/SAU 42',
    'Oyster River Cooperative School District':              'Oyster River Cooperative School District/SAU5',
    'Portsmouth School District':                            'Portsmouth School District/SAU52',
    'Rochester School District':                             'Rochester School District/SAU54',
    'Hopkinton School District':                            'Hopkinton School District/SAU66',
}


In [ ]:
import csv, re, json
from collections import defaultdict
from datetime import date, datetime
from openpyxl import load_workbook

ACK_DEADLINE_BD       = 5
FULFILL_DEADLINE_BD   = 25
IMMEDIATE_BD          = 10
DELAYED_FULFILLED_BD  = 60
AUTO_EXCLUDE_CATEGORIES   = {'RFP List'}
AUTO_EXCLUDE_STATUS_CODES = {'fix'}

COUNTERPART_OF = {
    'Berlin School District/SAU 3':                          'Berlin',
    'SAU6, serving the communities of Claremont and Unity':  'Claremont',
    'Concord School District/SAU8':                          'Concord',
    'Dover School District/SAU11':                           'Dover',
    'Franklin School District/SAU18':                        'Franklin',
    'Manchester School District':                            'Manchester',
    'Keene School District/SAU29':                           'Keene',
    'Laconia School District/SAU30':                         'Laconia',
    'Lebanon School District/SAU88':                         'Lebanon',
    'Nashua School District':                                'Nashua',
    'Portsmouth School District':                            'Portsmouth',
    'Rochester School District':                             'Rochester',
    'Somersworth School District/SAU56':                     'Somersworth',
    'Contoocook Valley School District/SAU1':                'Peterborough',
    'Shaker Regional School District/SAU80':                 'Belmont',
    'SAU70':                                                 'Hanover',
    'SAU20':                                                 'Gorham',
    'SAU67':                                                 'Bow',
    'Hopkinton School District':                             'Hopkinton',
    'Derry Cooperative School District':                     'Derry',
    'Salem School District/SAU57':                           'Salem',
    'Monadnock Regional School District':                    'Troy',
    'Oyster River Cooperative School District':              'Durham',
}

#we had a couple duplicate agencies in the project 

AGENCY_REMAP = {
    ('Lebanon', 'City Clerk'):        'Lebanon City Clerk',
    ('Lebanon', 'Lebanon Clerk'):     'Lebanon City Clerk',
    ('Nashua',  'Nashua City Clerk'): 'Nashua City Clerk',
    ('Nashua',  'Nashua Clerk'):      'Nashua City Clerk',
}

RAW_DISTRICT_NAMES = set(COUNTERPART_OF.keys())

def display_of(raw_name):
    return DISTRICT_DISPLAY_NAME.get(raw_name, raw_name)

DISTRICT_DISPLAY_NAMES   = {display_of(raw) for raw in RAW_DISTRICT_NAMES}
DISTRICT_TO_HOST_TOWN    = {display_of(raw): host for raw, host in COUNTERPART_OF.items()}
TOWN_TO_DISTRICT_DISPLAY = {host: display_of(raw) for raw, host in COUNTERPART_OF.items()}

_TRAIL_PAREN = __import__("re").compile(r"\s*\([^)]*\)\s*$")
def category(t): return _TRAIL_PAREN.sub("", t or "").strip()

def entity_of(row):
    a = (row.get('Agency Name') or '').strip()
    if a in RAW_DISTRICT_NAMES:
        return display_of(a)
    return row['Jurisdiction']

def kind_of(entity):
    return 'district' if entity in DISTRICT_DISPLAY_NAMES else 'town'

def counterpart_of(entity):
    if entity in DISTRICT_DISPLAY_NAMES:
        return DISTRICT_TO_HOST_TOWN[entity]
    return TOWN_TO_DISTRICT_DISPLAY.get(entity)

def display_agency(row):
    e = entity_of(row)
    a = (row.get('Agency Name') or '').strip()
    cleaned = AGENCY_REMAP.get((e, a), a)
    return DISTRICT_DISPLAY_NAME.get(cleaned, cleaned)

def normalized_tags(r): return (r.get('Tags') or '').lower().replace('nh-in-person', 'nh-inperson')
def has_tag(r, frag): return frag in normalized_tags(r)

def _parse_dt(s):
    if not s: return None
    try: return datetime.strptime(s.split()[0], '%Y-%m-%d').date()
    except ValueError: return None

def business_days_between(a, b):
    if not a or not b or b <= a: return 0
    total = (b - a).days
    weeks, rem = divmod(total, 7)
    n = weeks * 5
    wd = a.weekday()
    for d in range(1, rem + 1):
        if (wd + d) % 7 < 5: n += 1
    return n

def business_days_open(row, today=None):
    s = _parse_dt(row.get('Date Submitted'))
    d = _parse_dt(row.get('Date Done'))
    return business_days_between(s, d or (today or date.today()))

def auto_score(row, today=None):
    code = (row.get('Status Code') or '').strip()
    bd   = business_days_open(row, today)
    if has_tag(row, 'privacy'): return -2
    if code == 'done':
        if has_tag(row, 'nh-inperson') or has_tag(row, 'nh-citizen'): return 2
        if bd > DELAYED_FULFILLED_BD: return 3
        if bd <= IMMEDIATE_BD: return 5
        return 4
    if code == 'no_docs':  return 0
    if code == 'fix':      return None
    if code == 'partial':  return None
    if code == 'rejected': return -4
    if code == 'ack':                          return -5 if bd > ACK_DEADLINE_BD else None
    if code in ('processed', 'submitted'):     return -3 if bd > FULFILL_DEADLINE_BD else None
    return None



In [ ]:
with open(RESULTS_CSV, newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

wb = load_workbook(SCORECARD_XLSX, data_only=False)  
req_ws = wb['Requests']
ent_ws = wb['Entities']

req_headers = [c.value for c in req_ws[1]]
H = {h: i for i, h in enumerate(req_headers)}

req_overrides = {}
for r in req_ws.iter_rows(min_row=2, values_only=True):
    if not r or r[H['Request ID']] in (None, ''): continue
    rid = str(r[H['Request ID']])
    man  = r[H['Manual Score']]
    excl = r[H['Exclude']]
    notes = r[H['Notes']]
    req_overrides[rid] = {
        'manual':  float(man) if isinstance(man, (int, float)) else None,
        'exclude': bool(excl),
        'notes':   (notes or '').strip() if isinstance(notes, str) else '',
    }

entity_editorial = {}
for r in ent_ws.iter_rows(min_row=2, values_only=True):
    if not r or not r[0]: continue
    entity_editorial[r[0]] = {
        'summary':        (r[5] or '').strip() if len(r) > 5 and isinstance(r[5], str) else '',
        'agencyResponse': (r[6] or '').strip() if len(r) > 6 and isinstance(r[6], str) else '',
    }

print(f'Loaded {len(rows)} requests, {len(req_overrides)} request override rows, {len(entity_editorial)} entity rows.')
n_with_summary = sum(1 for v in entity_editorial.values() if v['summary'])
n_with_response = sum(1 for v in entity_editorial.values() if v['agencyResponse'])
print(f'Entities with manual summary: {n_with_summary} / {len(entity_editorial)}')
print(f'Entities with manual Agency Response: {n_with_response} / {len(entity_editorial)}')


In [ ]:
today = date.today()

def effective_score(row, override):
    if override.get('exclude'): return None
    if override.get('manual') is not None:
        return float(override['manual'])
    return auto_score(row, today)

by_entity = defaultdict(list)
for r in rows:
    by_entity[entity_of(r)].append(r)

entity_stats = {}
RESPONDED_CODES = {'done', 'no_docs', 'partial', 'rejected', 'fix'}

for entity, reqs in by_entity.items():
    scores       = []
    request_items = []
    for r in reqs:
        rid = str(r['Request ID'])
        ov  = req_overrides.get(rid, {'manual': None, 'exclude': False, 'notes': ''})
        eff = effective_score(r, ov)
        if eff is not None:
            scores.append(eff)
        code = (r.get('Status Code') or '').strip()
        request_items.append({
            'id':       rid,
            'title':    category(r['Title']),
            'agency':   display_agency(r),
            'status':   r.get('Status') or '',
            'code':     code,
            'score':    eff,
            'url':      r.get('MuckRock URL') or '',
            'excluded': ov['exclude'],
            'notes':    ov['notes'],
        })

    avg       = round(sum(scores) / len(scores), 2) if scores else None
    total     = len(reqs)
    responded = sum(1 for r in reqs if (r.get('Status Code') or '').strip() in RESPONDED_CODES)
    rejected  = sum(1 for r in reqs if (r.get('Status Code') or '').strip() == 'rejected')

    done_reqs = [r for r in reqs if (r.get('Status Code') or '').strip() == 'done']
    if any(has_tag(r, 'nh-inperson') or has_tag(r, 'nh-citizen') for r in done_reqs):
        records = 'in-person'
    elif done_reqs:
        records = 'digital'
    else:
        records = 'none'

    reply_bds = [business_days_open(r, today) for r in reqs
                  if (r.get('Status Code') or '').strip() in RESPONDED_CODES
                  and business_days_open(r, today) > 0]
    avg_reply = f'{round(sum(reply_bds) / len(reply_bds))} days' if reply_bds else '—'

    missed = any(
        ((r.get('Status Code') or '').strip() == 'ack' and business_days_open(r, today) > ACK_DEADLINE_BD)
        or ((r.get('Status Code') or '').strip() in ('processed', 'submitted') and business_days_open(r, today) > FULFILL_DEADLINE_BD)
        for r in reqs
    )

    editorial   = entity_editorial.get(entity, {})
    summary     = editorial.get('summary') or default_summary(entity, avg)
    agency_resp = editorial.get('agencyResponse') or None

    entity_stats[entity] = {
        'kind':            kind_of(entity),
        'counterpart':     counterpart_of(entity),
        'score':           avg if avg is not None else 0,
        'requests':        total,
        'responded':       responded,
        'graded':          len(scores),
        'avgReply':        avg_reply,
        'records':         records,
        'rejected':        rejected,
        'missedDeadline':  bool(missed),
        'summary':         summary,
        'agencyResponse':  agency_resp,
        'requestList':     request_items,
    }

# prev
print(f'{"KIND":<9}{"ENTITY":<42}{"SCORE":>7}{"REQ":>5}{"GRD":>5}{"REJ":>5}{"REPLY":>10}')
for e in sorted(entity_stats, key=lambda k: (entity_stats[k]['kind'], -entity_stats[k]['score'])):
    s = entity_stats[e]
    print(f"{s['kind']:<9}{e:<42}{s['score']:+6.2f}{s['requests']:>5}{s['graded']:>5}{s['rejected']:>5}{s['avgReply']:>10}")



In [ ]:
def js_str(s): return json.dumps(s, ensure_ascii=False)
def js_bool(b): return 'true' if b else 'false'

def render_block(varname, kind_filter):
    lines = [f'const {varname} = {{']
    items = [(e, s) for e, s in entity_stats.items() if s['kind'] == kind_filter]
    for e, s in sorted(items, key=lambda kv: kv[0]):
        lines.append(f'  {js_str(e)}: {{')
        lines.append(f'    counterpart:    {js_str(s["counterpart"]) if s["counterpart"] else "null"},')
        lines.append(f'    score:          {s["score"]},')
        lines.append(f'    requests:       {s["requests"]},')
        lines.append(f'    responded:      {s["responded"]},')
        lines.append(f'    graded:         {s["graded"]},')
        lines.append(f'    avgReply:       {js_str(s["avgReply"])},')
        lines.append(f'    records:        {js_str(s["records"])},')
        lines.append(f'    rejected:       {s["rejected"]},')
        lines.append(f'    missedDeadline: {js_bool(s["missedDeadline"])},')
        lines.append(f'    summary:        {js_str(s["summary"])},')
        ar = s['agencyResponse']
        lines.append(f'    agencyResponse: {js_str(ar) if ar else "null"},')
        lines.append(f'    requestList: [')
        for req in s['requestList']:
            score_str = json.dumps(req['score']) if req['score'] is not None else 'null'
            lines.append(
                '      { '
                f'id: {js_str(req["id"])}, '
                f'title: {js_str(req["title"])}, '
                f'agency: {js_str(req["agency"])}, '
                f'status: {js_str(req["status"])}, '
                f'code: {js_str(req["code"])}, '
                f'score: {score_str}, '
                f'url: {js_str(req["url"])}, '
                f'excluded: {js_bool(req["excluded"])}, '
                f'notes: {js_str(req["notes"])} '
                '},'
            )
        lines.append(f'    ],')
        lines.append(f'  }},')
    lines.append('};')
    return '\n'.join(lines)

town_block     = render_block('TOWN_OVERRIDES',     'town')
district_block = render_block('DISTRICT_OVERRIDES', 'district')

print(town_block[:800])
print('…')
print(district_block[:500])



In [ ]:
with open(INDEX_HTML, 'r', encoding='utf-8') as f:
    html = f.read()

town_pat     = re.compile(r'const TOWN_OVERRIDES = \{[\s\S]*?\n\};', re.MULTILINE)
district_pat = re.compile(r'const DISTRICT_OVERRIDES = \{[\s\S]*?\n\};', re.MULTILINE)

if not town_pat.search(html):
    raise RuntimeError(f'Could not find `const TOWN_OVERRIDES = {{ ... }};` in {INDEX_HTML}')
if not district_pat.search(html):
    raise RuntimeError(f'Could not find `const DISTRICT_OVERRIDES = {{ ... }};` in {INDEX_HTML}')

html, nt = town_pat.subn(town_block, html, count=1)
html, nd = district_pat.subn(district_block, html, count=1)

with open(INDEX_HTML, 'w', encoding='utf-8') as f:
    f.write(html)

n_towns     = sum(1 for s in entity_stats.values() if s['kind'] == 'town')
n_districts = sum(1 for s in entity_stats.values() if s['kind'] == 'district')
total_reqs  = sum(len(s['requestList']) for s in entity_stats.values())
print(f'✓ updated {INDEX_HTML}: TOWN_OVERRIDES ({n_towns}), DISTRICT_OVERRIDES ({n_districts}), {total_reqs} requests embedded')

